# 1.加载模型

In [4]:
from transformers import TextStreamer
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "/Users/wjx/Desktop/modles/Qwen3.5-0.8B"

model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="mps")
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
print(next(model.parameters()).device)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

mps:0


In [5]:
# 根据用户的提问内容生成模型的单次回答
def get_response(prompt, generation_params={}):
    # instruction = """[INST] <<SYS>>\nYou are a helpful, respectful and honest assistant. Always answer as helpfu

    #     If a question does not make any sense, or is not factually coherent, explain why instead of answerin
    # prompt = instruction.format(prompt)
    messages = [
        {"role": "system", "content": "你是一个智能助理，始终用中文回答用户的问题"},
        {"role": "user", "content": prompt}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    print(prompt)
    model_inputs = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(model.device)

    # 生成模型的回复结果
    generated_ids = model.generate(**model_inputs, max_new_tokens=4096, **generation_params)
    # 模型的返回内容包括输入文本和输出结果，这里仅取出模型生成的输出结果
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in
                     zip(model_inputs.input_ids, generated_ids)]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response


message = '''你是谁'''
params = {
    "do_sample": True,
    "temperature": 0.7,
}
print(get_response(message, params))

<|im_start|>system
你是一个智能助理，始终用中文回答用户的问题<|im_end|>
<|im_start|>user
你是谁<|im_end|>
<|im_start|>assistant
<think>

</think>


我是 Qwen3.5，阿里巴巴通义实验室自主研发的超大规模语言模型。我专注于人工智能领域的专业技术问题、创意写作、代码开发、逻辑推理等多个领域。如果您有任何具体问题或需要帮助，请随时告诉我！



# 思维链介绍

思维链（Chain of Thought，简称 CoT）是一种提示人工智能模型进行推理的方法，通过引导模型像人类一样逐步思考来解决复杂问题。让我通过具体例子说明。像人类解决数学题时会写出计算过程，思维链也是类似的原理。比如解决这个问题："一个苹果 5 元，梨 3 元，如果买 2 个苹果和 3 个梨一共需要多少钱?"使用思维链的思考过程是：

先计算苹果的总价: 2 个苹果 × 5 元 = 10 元 再计算梨的总价: 3 个梨 × 3 元 = 9 元 最后计算总和: 10 元 + 9 元 = 19 元

这种一步步展现推理过程的方法，不仅能帮助模型得出更准确的答案，还能让人理解模型是如何得出结论的。这比直接给出答案"19元"要更可靠和透明。思维链在解决数学题、逻辑推理、多步骤任务等场景特别有效。它是提升AI模型表现的重要技术之一。

# 2.思维链的实现

In [ ]:
# 让模型直接给出答案

prompt1 = '''直接回答以下问题，输出一个实体或数字。
问: 小明的爸爸有两个儿子，一个叫小红，另一个叫什么?
答: 小明
问: 有 10 个朋友在线玩电子游戏，其中 7 个玩家退出。如果每个离开的玩家都有 8 条命，那么他们总共有多少条命?
答: 24
问: 每本杂志成本 3 美元。Jewel 买了 10 本杂志，决定每本售价 3.50 美元。但他父亲建议他涨价1.50美元，他同意了。Jewel 的邻居买了 1000 份报纸。Jewel 卖掉杂志能赚多少钱
答:
'''

# 让模型一步一步思考给出答案

prompt2 = '''问: 每本杂志成本 3 美元。Jewel 买了 10 本杂志，决定每本售价 3.50 美元。但他父亲建议他涨价1.50美元，他同意了。Jewel 的邻居买了 1000 份报纸。Jewe卖掉杂志可以获得多少钱
答: 让我们一步一步思考, '''

print(get_response(prompt1))
print("-" * 100)
print(get_response(prompt2))

<|im_start|>system
你是一个智能助理，始终用中文回答用户的问题<|im_end|>
<|im_start|>user
直接回答以下问题，输出一个实体或数字。
问: 小明的爸爸有两个儿子，一个叫小红，另一个叫什么?
答: 小明
问: 有 10 个朋友在线玩电子游戏，其中 7 个玩家退出。如果每个离开的玩家都有 8 条命，那么他们总共有多少条命?
答: 24
问: 每本杂志成本 3 美元。Jewel 买了 10 本杂志，决定每本售价 3.50 美元。但他父亲建议他涨价1.50美元，他同意了。Jewel 的邻居买了 1000 份报纸。Jewel 卖掉杂志能赚多少钱
答:<|im_end|>
<|im_start|>assistant
<think>

</think>


150 美元

----------------------------------------------------------------------------------------------------
<|im_start|>system
你是一个智能助理，始终用中文回答用户的问题<|im_end|>
<|im_start|>user
问: 每本杂志成本 3 美元。Jewel 买了 10 本杂志，决定每本售价 3.50 美元。但他父亲建议他涨价1.50美元，他同意了。Jewel 的邻居买了 1000 份报纸。Jewe卖掉杂志可以获得多少钱
答: 让我们一步一步思考,<|im_end|>
<|im_start|>assistant
<think>

</think>




结论：让模型一步一步思考可以提高模型解决数学问题的水平

In [ ]:
# 特别是，在需要分多步调用外部工具的时候，CoT更为必要。

prompt1 = '''分步回答一下问题，如果某些步骤需要调用检索器，请输出[R]+检索内容。
问: 2015 年 Diamond Head Classic 的 MVP 球员效力于哪支球队?
答: 让我们一步一步思考,
'''

print(get_response(prompt1))

# 3.思维链的构造与使用
- 在实际使用的时候，我们常常需要利用少样本提示，教会模型在一个新的任务上的、新的推理结构，以便模型按照我们希望的方式，进行推理。
- 这有利于后续的推理步骤解析，进而有利于推理逻辑校验、修正、可解释。
- 这就需要我们对思维链进行构造，创造出思维链的提示样本，即演示。

- Manual-CoT: 遵循传统的手动样本生成方法，将生成的推理链作为示例。
- Retrieve-CoT: 检索与所给问题最相近的例子作为演示，并使用 Zero-Shot-CoT 生成其推理链，将生成的推理链作为示例。
- Auto-CoT: 包括两个主要阶段：(i) 问题聚类：将给定的问题数据集分成几个聚类；(ii) 演示抽样：从每个聚类中选择一个代表性问题，并使用 Zero-Shot-CoT 生成其推理链，将生成的推理链作为示例。

In [ ]:
# Manual-CoT: 遵循传统的手动样本生成方法，将生成的推理链作为示例。
prompt1 = '''
问: B is not A.
So if A is True,
is B True or False?
答: 因为"A是True"和"B是非A", 所以B是False。
问: B is not A.
C is not B.
So if A is True,
is C True or False?
答: 因为"A是True"和"B是非A", 所以B是False。
因为"B是False"和"C是非B", 所以C是True。
问: B is not A.
C is not B.
D is not C.
So if A is True,
is D True or False?
答:
'''
# 可以看到推理结构被改变了，遵循了我们演示的推理结构。
print(get_response(prompt1))

In [ ]:
# Retrieve-CoT
# 候选数据集，每次思维链选择最相近的
data = [
    'B是A取反。那么，如果A是真，B是真还是假？',
    '小明的爸爸有两个儿子，一个叫小红，另一个叫什么？',
    '每本杂志成本 3 美元。Jewel 买了 10 本杂志，决定每本售价 3.50 美元。但他父亲建议他涨价1.50美元，他同意了。Jewel 的邻居买了 1000 份报纸。Jewel 卖掉杂志能赚多少钱？'
]
question = '''B是A取反。C是B取反。D是C取反。那么，如果A是真，D是真还是假？
答:
'''

# 给定数据集，选出与问题最相关的样本
def levenshtein_distance(s1, s2):
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)

    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row

    return previous_row[-1]

# 与问题最相关的样本
distance = []
for i in range(len(data)):
    distance.append(levenshtein_distance(data[i], question))
demo = data[distance.index(min(distance))]

# 利用与问题最相关的样本，通过zeroshot的方式，利用大模型本身的能力，自动化构造思维链演示。
prompt1 = '问: ' + demo + '\n答:'
demo_ans = get_response(demo)
print(demo_ans)

# 将该演示作为少样本学习的演示。也可以保存并复用。
prompt2 = '问: ' + demo + '\n答:' + demo_ans + '\n' + '问: ' + question + '\n答:'
print(get_response(prompt2))